In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e7/train.csv
/kaggle/input/competitions/playground-series-s6e7/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/sample_submission.csv")

In [3]:
sample.tail()

,id,health_condition
295748,985836,at-risk
295749,985837,at-risk
295750,985838,at-risk
295751,985839,at-risk
295752,985840,at-risk


In [4]:
train.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,NaN,average,sedentary,NaN,male


In [5]:
train.shape

(690088, 15)

In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       690088 non-null  int64  
 1   health_condition         690088 non-null  object 
 2   sleep_duration           614089 non-null  float64
 3   heart_rate               682255 non-null  float64
 4   bmi                      676190 non-null  float64
 5   calorie_expenditure      637235 non-null  float64
 6   step_count               676172 non-null  float64
 7   exercise_duration        683187 non-null  float64
 8   water_intake             646611 non-null  float64
 9   diet_type                683187 non-null  object 
 10  stress_level             607277 non-null  object 
 11  sleep_quality            631757 non-null  object 
 12  physical_activity_level  653467 non-null  object 
 13  smoking_alcohol          661506 non-null  object 
 14  gend

In [7]:
train.isna().sum()

id                             0
health_condition               0
sleep_duration             75999
heart_rate                  7833
bmi                        13898
calorie_expenditure        52853
step_count                 13916
exercise_duration           6901
water_intake               43477
diet_type                   6901
stress_level               82811
sleep_quality              58331
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
dtype: int64

In [8]:
num_cols = train.select_dtypes(include=['int64','float64']).columns

for col in num_cols:
    train[col]=train[col].fillna(train[col].median())

In [9]:
cat_cols = train.select_dtypes(include=['object']).columns

for col in cat_cols:
    train[col] = train[col].fillna(train[col].mode()[0])

In [10]:
train.isna().sum()

id                         0
health_condition           0
sleep_duration             0
heart_rate                 0
bmi                        0
calorie_expenditure        0
step_count                 0
exercise_duration          0
water_intake               0
diet_type                  0
stress_level               0
sleep_quality              0
physical_activity_level    0
smoking_alcohol            0
gender                     0
dtype: int64

In [11]:
for col in train.select_dtypes(include='object').columns:
    print(col,train[col].nunique())

health_condition 3
diet_type 3
stress_level 3
sleep_quality 3
physical_activity_level 3
smoking_alcohol 3
gender 3


all these cols have not that much categories so we don't need to drop it.

In [12]:
train.drop(columns='id',inplace=True)

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           690088 non-null  float64
 2   heart_rate               690088 non-null  float64
 3   bmi                      690088 non-null  float64
 4   calorie_expenditure      690088 non-null  float64
 5   step_count               690088 non-null  float64
 6   exercise_duration        690088 non-null  float64
 7   water_intake             690088 non-null  float64
 8   diet_type                690088 non-null  object 
 9   stress_level             690088 non-null  object 
 10  sleep_quality            690088 non-null  object 
 11  physical_activity_level  690088 non-null  object 
 12  smoking_alcohol          690088 non-null  object 
 13  gender                   690088 non-null  object 
dtypes: f

In [14]:
from sklearn.preprocessing import LabelEncoder

cat_cols = train.select_dtypes(include=["object"]).columns

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col].astype(str))
    encoders[col] = le

In [15]:
for col, le in encoders.items():
    print(f"\nColumn: {col}")
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(mapping)


Column: health_condition
{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}

Column: diet_type
{'balanced': np.int64(0), 'non-veg': np.int64(1), 'veg': np.int64(2)}

Column: stress_level
{'high': np.int64(0), 'low': np.int64(1), 'medium': np.int64(2)}

Column: sleep_quality
{'average': np.int64(0), 'good': np.int64(1), 'poor': np.int64(2)}

Column: physical_activity_level
{'active': np.int64(0), 'moderate': np.int64(1), 'sedentary': np.int64(2)}

Column: smoking_alcohol
{'no': np.int64(0), 'occasional': np.int64(1), 'yes': np.int64(2)}

Column: gender
{'female': np.int64(0), 'male': np.int64(1), 'other': np.int64(2)}


In [16]:
train.head()

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,2,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,2,0,0,2,2,0
1,0,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,1,1,0,1,2,2
2,2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,2,0,2,0,2,1
3,2,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,2,0,0,0,1,0
4,0,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,2,2,0,2,2,1


In [17]:
X = train.drop('health_condition',axis=1)
y = train['health_condition']

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
print(X_train.shape)
print(X_val.shape)

(552070, 13)
(138018, 13)


## XGBoost

In [20]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import StratifiedKFold, cross_val_score

# XGBmodel = XGBClassifier(
#     n_estimators=500,
#     learning_rate=0.05,
#     max_depth=6,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42,
#     eval_metric='mlogloss'
# )

In [21]:
# cv = StratifiedKFold(
#     n_splits=5,
#     shuffle=True,
#     random_state=42
# )

# scores = cross_val_score(
#     XGBmodel,
#     X,
#     y,
#     cv=cv,
#     scoring='accuracy',
#     n_jobs=-1
# )

# print("Fold Accuracies:", scores)
# print("Mean Accuracy:", scores.mean())
# print("Standard Deviation:", scores.std())

In [22]:
# XGBmodel.fit(X, y)

## LightBGM

In [23]:
from lightgbm import LGBMClassifier

In [24]:
lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

lgbm.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014429 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1802
[LightGBM] [Info] Number of data points in the train set: 552070, number of used features: 13
[LightGBM] [Info] Start training from score -0.152594
[LightGBM] [Info] Start training from score -2.852826
[LightGBM] [Info] Start training from score -2.478835


LGBMClassifier(learning_rate=0.05, n_estimators=500, random_state=42)

In [25]:
pred = lgbm.predict(X_val)

In [26]:
from sklearn.metrics import balanced_accuracy_score

score = balanced_accuracy_score(y_val, pred)
print(score)

0.8576648393924282


## RandomForestClassifier

In [27]:
# from sklearn.ensemble import RandomForestClassifier

# rf = RandomForestClassifier(
#     n_estimators=200,
#     random_state=42,
#     n_jobs=-1
# )

# rf.fit(X_train,y_train)

In [28]:
# pred = rf.predict(X_val)

In [29]:
# from sklearn.metrics import balanced_accuracy_score

# score = balanced_accuracy_score(y_val, pred)

# print("Balanced Accuracy:", score)

In [30]:
# feature_importance = (
#     pd.DataFrame({
#         'feature': X.columns,
#         'importance': rf.feature_importances_
#     })
#     .sort_values('importance', ascending=False)
# )

# feature_importance.tail(10)

## Test 

In [31]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       295753 non-null  int64  
 1   sleep_duration           263182 non-null  float64
 2   heart_rate               292396 non-null  float64
 3   bmi                      289797 non-null  float64
 4   calorie_expenditure      273101 non-null  float64
 5   step_count               289789 non-null  float64
 6   exercise_duration        292795 non-null  float64
 7   water_intake             277120 non-null  float64
 8   diet_type                292795 non-null  object 
 9   stress_level             260263 non-null  object 
 10  sleep_quality            270754 non-null  object 
 11  physical_activity_level  280058 non-null  object 
 12  smoking_alcohol          283504 non-null  object 
 13  gender                   286593 non-null  object 
dtypes: f

In [32]:
test.shape

(295753, 14)

In [33]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,NaN
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [34]:
test.isna().sum()

id                             0
sleep_duration             32571
heart_rate                  3357
bmi                         5956
calorie_expenditure        22652
step_count                  5964
exercise_duration           2958
water_intake               18633
diet_type                   2958
stress_level               35490
sleep_quality              24999
physical_activity_level    15695
smoking_alcohol            12249
gender                      9160
dtype: int64

In [35]:
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

for col in num_cols:
    test[col]= test[col].fillna(X[col].median())

for col in cat_cols:
    test[col] = test[col].fillna(X[col].mode()[0])

In [36]:
test.isna().sum().sum()

np.int64(0)

In [37]:
test.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,1.0
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [38]:
student_ids = test['id'].copy()

In [39]:
test.drop(columns='id',inplace=True)

In [40]:
# diet_type
test['diet_type'] = test['diet_type'].map({
    'balanced': 0,
    'non-veg': 1,
    'veg': 2
})

# stress_level
test['stress_level'] = test['stress_level'].map({
    'high': 0,
    'low': 1,
    'medium': 2
})

# sleep_quality
test['sleep_quality'] = test['sleep_quality'].map({
    'average': 0,
    'good': 1,
    'poor': 2
})

# physical_activity_level
test['physical_activity_level'] = test['physical_activity_level'].map({
    'active': 0,
    'moderate': 1,
    'sedentary': 2
})

# smoking_alcohol
test['smoking_alcohol'] = test['smoking_alcohol'].map({
    'no': 0,
    'occasional': 1,
    'yes': 2
})

# gender
test['gender'] = test['gender'].map({
    'female': 0,
    'male': 1,
    'other': 2
})

In [41]:
print(test.dtypes)
print(test.select_dtypes(include='object').columns)

sleep_duration             float64
heart_rate                 float64
bmi                        float64
calorie_expenditure        float64
step_count                 float64
exercise_duration          float64
water_intake               float64
diet_type                  float64
stress_level               float64
sleep_quality              float64
physical_activity_level    float64
smoking_alcohol            float64
gender                     float64
dtype: object
Index([], dtype='object')


In [42]:
test_pred = lgbm.predict(test)

In [43]:
test.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
0,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,2.0,0.0,2.0,0.0,1.0,1.0
1,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,0.0,0.0,2.0,2.0,2.0,2.0
2,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,0.0,2.0,2.0,0.0,0.0,NaN
3,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,2.0,1.0,1.0,1.0,2.0,2.0
4,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,2.0,0.0,0.0,0.0,1.0,2.0


In [44]:
test_pred[:10]

array([2, 0, 0, 0, 2, 0, 0, 0, 0, 0])

In [45]:
submission = pd.DataFrame({
    "id": student_ids,
    "health_condition": test_pred
})

In [46]:
submission.head(10)

,id,health_condition
0,690088,2
1,690089,0
2,690090,0
3,690091,0
4,690092,2
5,690093,0
6,690094,0
7,690095,0
8,690096,0
9,690097,0


In [47]:
submission.isna().sum()

id                  0
health_condition    0
dtype: int64

Column: health_condition
{'at-risk': np.int64(0), 'fit': np.int64(1), 'unhealthy': np.int64(2)}

In [48]:
submission["health_condition"] = submission["health_condition"].astype(int)

mapping = {
    0: "at-risk",
    1: "fit",
    2: "unhealthy"
}

submission["health_condition"] = submission["health_condition"].map(mapping)

In [49]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,at-risk
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy


In [50]:
submission.to_csv("submission.csv", index=False)